# Load the venv after setup

In [1]:
# Select the kernel with magenta_venv (top right dropdown).
# This should print: /teamspace/studios/this_studio/magenta_venv/bin/python
import sys
print(sys.executable)

/teamspace/studios/this_studio/magenta_venv/bin/python


# Imports

In [2]:
import asyncio
import asyncio.runners
import gradio as gr
import soundfile as sf
import io
import websockets
import json
import threading
import time
import subprocess
import librosa
import numpy as np

asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

# Configuration and variables

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Maps each emotion label to the musical descriptors used to steer
# Magenta RT's text-prompt embedding when that emotion is detected.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EMOTION_PROMPTS = {
    "joy":          ["bright pads", "major harmony"],
    "anger":        ["distorted pad"],
    "annoyance":    ["noisy textures"],
    "disapproval":  ["dissonant melody"],
    "disgust":      ["sub bass pulse", "chaotic textures", "organic timbre"],
    "embarrassment":["uneasy textures"],
    "fear":         ["creepy sounds"],
    "grief":        ["hollow pad", "melancholic melody"],
    "nervousness":  ["trembling drone", "unstable textures"],
    "remorse":      ["reverse pads", "detuned textures"],
    "sadness":      ["slow melody", "minor pads"],

    "admiration":   ["slow counterpoint", "shimmering textures"],
    "amusement":    ["playful plucks"],
    "approval":     ["resolved harmony"],
    "desire":       ["lush pad", "gliding melody"],
    "caring":       ["warm textures", "soft pad"],
    "excitement":   ["glittering textures"],
    "gratitude":    ["dreamy pad", "ethereal textures"],
    "love":         ["warm timbre", "love pad"],
    "optimism":     ["bright suspended chords"],
    "pride":        ["wide drone", "saw timbres"],

    "relief":       ["relaxing pads", "soft timbre"],
    "confusion":    ["shepard tones", "microtones"],
    "curiosity":    ["glittering arpeggios"],
    "realization":  ["tonic drone"],
    "surprise":     ["sudden motifs", "accented textures"],
}

# NUM_TEXT_PROMPTS = total keywords above (42) + "ambient" + "no drums" (see current_params below).
NUM_TEXT_PROMPTS  = 44
NUM_AUDIO_PROMPTS = 10

In [5]:
# List of emotions used for the hardcoded audio-prompt slots (see below)
# and for the "_<emotion>_audio_" keys sent over the prompt websocket.
emotions = [
    "anger",
    "fear",
    "grief",
    "joy",
    "love",
    "neutral",
    "optimism",
    "remorse",
    "sadness",
    "caring",
]

# Maps each websocket key (e.g. "_anger_audio_") to its index in
# initial_audio_prompts / current_params["audio_prompts"].
audio_key_to_index = {
    f"_{emotion}_audio_": i
    for i, emotion in enumerate(emotions)
}

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Load one hardcoded reference audio file per emotion.
# Replace the files in `audio_folder` with your own recordings if needed
# (each file must be named "<emotion>_audio.wav").
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_hardcoded_audio(filepath, weight=1.0):
    """Loads an audio file and formats it to be compatible with Magenta RT."""
    # Resample to 16000 Hz (Magenta RT's expected sampling rate).
    # librosa.load automatically converts to mono and returns float32 samples in [-1.0, 1.0].
    data, sr = librosa.load(filepath, sr=16000)

    # Keep only the first 10 seconds.
    data = data[:sr * 10]

    return (data, weight)


audio_folder = "./test_audio"
initial_audio_prompts = [None] * len(emotions)

for i, emotion in enumerate(emotions):
    file_path = f"{audio_folder}/{emotion}_audio.wav"
    try:
        # "neutral" gets a small starting weight, so the model has a
        # baseline style active before any emotion-driven prompt arrives.
        if emotion == "neutral":
            initial_audio_prompts[i] = load_hardcoded_audio(file_path, weight=0.3)
        else:
            initial_audio_prompts[i] = load_hardcoded_audio(file_path, weight=0.0)
        print(f"Loaded: {emotion}")
    except Exception as e:
        print(f"Error when loading {emotion} audio file: {e}")

Loaded: anger
Loaded: fear
Loaded: grief
Loaded: joy
Loaded: love
Loaded: neutral
Loaded: optimism
Loaded: remorse
Loaded: sadness
Loaded: caring


In [7]:
# Global mutable state shared between the Gradio UI, the websocket
# servers, and the generation loop. Every component reads/writes here
# directly (no locks -- fields are simple list/dict assignments).
state = None
current_params = {
    "temperature": 0.8, 
    "topk": 20,
    "guidance": 2.5,
    "prompts": [
        (prompt, 0.0)
        for prompt_list in EMOTION_PROMPTS.values()
        for prompt in prompt_list
    ] + [("ambient", 1.0)] + [("no drums", 1.0)],
    "audio_prompts": initial_audio_prompts,
}

TEMPERATURE: 
This controls how chaotic the model behaves. Low temperature values (e.g., 0.9) will make the model's choices more predictable and stable. High values (e.g., 1.5) will encourage more surprising and experimental musical ideas, but can also lead to instability.

TOPK: 
This parameter filters the model's vocabulary at each step. It forces the model to choose its next prediction only from the k most likely options.
A low topk value (e.g., 40) restricts the model to a smaller, safer palette of options. This leads to more coherent and predictable music that is less likely to have dissonant errors, but can sometimes feel repetitive.
A high topk value gives the model a much wider range of choices, allowing for more variety and unexpected turns. This can make the output more creative, but also noisier.

GUIDANCE: 
This controls how strictly the generated music should adhere to the text prompts.
A higher value will push the model to produce a textbook example of the chosen style, emphasizing its key characteristics.
A lower value will treat the text prompts more as a loose inspiration, allowing the model more creative freedom to deviate and blend other influences.


# Streaming music generation
The section that generates and transmits real-time synthesized audio from Magenta RT to the browser. It produces float32 PCM at 48kHz stereo and sends it via WebSocket as a continuous stream.

DISCLAIMER

Magenta RT's training data primarily consists of Western instrumental music. As a consequence, Magenta RT has incomplete coverage of both vocal performance and the broader landscape of rich musical traditions worldwide.


## Initialize model

In [8]:
import nest_asyncio
import os
nest_asyncio.apply()

In [9]:
from magenta_rt import system
# Fetch all assets from HuggingFace and initialize model (~2-5 minutes).
BACKEND = "Magenta RT (Open weights)"
MRT = system.MagentaRT(tag="large", lazy=False)

I0000 00:00:1782144718.132190   10621 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79187 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
E0622 16:12:18.726133   13106 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0622 16:12:19.097744   13107 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0622 16:12:25.785340   13110 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0622 16:12:26.189597   13104 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0

In [10]:
# nest_asyncio patched asyncio.run above to support nested event loops;
# restore the original implementation for the rest of the notebook.
asyncio.run = asyncio.runners.run

# Web Audio Socket
The browser-side receiver that consumes audio from the WebSocket stream. It feeds a ring buffer in memory, from which an AudioWorklet continuously extracts samples and plays them in the native AudioContext. This scheme keeps latency low and playback stable.

In [11]:
import queue as sync_queue

AUDIO_WS_PORT = 9004
_audio_q = sync_queue.Queue(maxsize=8)
_audio_ws_clients = set()
_audio_ws_loop = None

_flush_flag = threading.Event()

async def audio_ws_handler(websocket):
    """Handles one audio-stream websocket connection: seeds the client with
    a few chunks, then pulls one chunk from `_audio_q` each time the
    browser sends 'ready' (pull-based streaming, no fixed schedule)."""
    _audio_ws_clients.add(websocket)
    print(f"[Audio WS] Client connected: {websocket.remote_address}")

    async def get_next_chunk():
        while True:
            try:
                return _audio_q.get_nowait()
            except sync_queue.Empty:
                await asyncio.sleep(0.005)

    try:
        # Initial seed: 4 chunks (~8s of audio) before waiting for 'ready'.
        for _ in range(4):
            data = await get_next_chunk()
            await websocket.send(data)

        # Pull mode: the browser sends 'ready' when its buffer is low.
        async for message in websocket:
            if _flush_flag.is_set():
                _flush_flag.clear()
                await websocket.send(b"FLUSH")
                for _ in range(4):
                    data = await get_next_chunk()
                    await websocket.send(data)
                continue

            if message in (b'ready', 'ready'):
                data = await get_next_chunk()
                await websocket.send(data)

    except websockets.exceptions.ConnectionClosedError:
        pass
    finally:
        _audio_ws_clients.discard(websocket)
        print("[Audio WS] Client disconnected")

async def audio_ws_server():
    async with websockets.serve(audio_ws_handler, "0.0.0.0", AUDIO_WS_PORT, reuse_address=True):
        print(f"[Audio WS] Listening on :{AUDIO_WS_PORT}")
        await asyncio.Future()  # runs forever

def start_audio_ws_server():
    global _audio_ws_loop
    _audio_ws_loop = asyncio.new_event_loop()
    asyncio.set_event_loop(_audio_ws_loop)
    _audio_ws_loop.run_until_complete(audio_ws_server())


subprocess.run(f'fuser -k {AUDIO_WS_PORT}/tcp', shell=True, capture_output=True)
time.sleep(0.5)
threading.Thread(target=start_audio_ws_server, daemon=True).start()
print(f"Audio WS URL: wss://{AUDIO_WS_PORT}-{os.environ.get('LIGHTNING_CLOUDSPACE_HOST', 'localhost')}")

Audio WS URL: wss://9004-01kp66x818nvqtvyvcf9bkr5ze.cloudspaces.litng.ai
[Audio WS] Listening on :9004


## Play

### 1. UI update callbacks

In [12]:
import concurrent.futures
import functools

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Section 1 -- UI update callbacks
#
# Functions invoked by the Gradio components whenever the user
# interacts with the UI. They write directly into current_params.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def make_text_updater(i):
    """Returns a callback that updates the text of prompt slot `i`, keeping its weight unchanged."""
    def update(v): current_params["prompts"][i] = (v, current_params["prompts"][i][1])
    return update

def make_text_weight_updater(i):
    """Returns a callback that updates the weight of prompt slot `i`, keeping its text unchanged."""
    def update(v): current_params["prompts"][i] = (current_params["prompts"][i][0], v)
    return update

def make_audio_updater(i):
    """Returns a callback that updates the audio data of audio-prompt slot `i`.

    Normalizes the uploaded clip to mono float32 in [-1.0, 1.0], truncates it to
    10s, and invalidates the audio embedding cache so the new file gets re-embedded.
    """
    def update(v):
        if v is None:
            current_params["audio_prompts"][i] = (None, current_params["audio_prompts"][i][1])
            return
        sr, data = v
        data = data[:sr * 10]
        if data.ndim > 1:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        if data.max() > 1.0:
            data = data / 32768.0
        # ← FIX: resample a 16kHz prima di passare a MusicCoCa
        if sr != 16000:
            data = librosa.resample(data, orig_sr=sr, target_sr=16000)
        current_params["audio_prompts"][i] = (data, current_params["audio_prompts"][i][1])
        _embed_audio_cached.cache_clear()
    return update

def make_audio_weight_updater(i):
    """Returns a callback that updates the weight of audio-prompt slot `i`, keeping its data unchanged."""
    def update(v): current_params["audio_prompts"][i] = (current_params["audio_prompts"][i][0], v)
    return update

def update_temp(v):     current_params["temperature"] = v
def update_topk(v):     current_params["topk"] = v
def update_guidance(v): current_params["guidance"] = v

### 2. Style embedding

In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Section 2 -- Style embedding
#
# compute_style() runs on a dedicated thread via ThreadPoolExecutor,
# so it never blocks generation_loop.
#
# Text embeddings are cached with lru_cache: the same text always
# returns the same embedding immediately, without touching the GPU.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Thread pool dedicated to embedding, separate from the generation GPU work.
_style_executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
_style_future   = None    # Future for the style currently being computed
_current_style  = None    # Active style used by generation_loop

@functools.lru_cache(maxsize=256)
def _embed_text_cached(text):
    """Cached text embedding -- same string never re-queries the GPU."""
    return MRT.embed_style(text)

@functools.lru_cache(maxsize=16)
def _embed_audio_cached(audio_tuple):
    """Cached audio embedding -- same waveform never re-queries the GPU."""
    from magenta_rt import audio as audio_lib
    waveform = audio_lib.Waveform(np.asarray(audio_tuple), 16000)
    return MRT.embed_style(waveform)

def compute_style():
    """Computes the weighted-average style embedding from all active text and audio prompts."""
    weighted = np.zeros((768,), dtype=np.float32)
    total = 0.0
    for text, weight in current_params["prompts"]:
        if text.strip() and weight > 0.01:
            weighted += _embed_text_cached(text.strip()) * weight
            total += weight
    for audio_data, weight in current_params["audio_prompts"]:
        if audio_data is not None and weight > 0:
            # Convert to a tuple to make it hashable (required by lru_cache).
            weighted += _embed_audio_cached(tuple(audio_data.flatten().tolist())) * weight
            total += weight
    if total > 0:
        weighted /= total
    return weighted

### 3. Generation loop

In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Section 3 -- Generation loop
#
# Pattern identical to the official Magenta RT code:
#   - The style is recomputed in the background (ThreadPoolExecutor)
#   - generation_loop never waits for the new embedding: if the
#     Future isn't ready yet, it keeps using the previous style
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

stop_event = threading.Event()
gen_thread  = None

def generation_loop():
    """Main generation loop. Runs on its own thread, generating one audio
    chunk at a time and pushing it onto `_audio_q` for the audio websocket
    to consume, until `stop_event` is set."""
    global state, _current_style, _style_future
    state        = None
    last_prompts = None
    last_audio   = None

    # Compute the initial style synchronously before entering the loop,
    # so the first chunk already has a valid style.
    _current_style = compute_style()
    last_prompts   = list(current_params["prompts"])
    last_audio     = list(current_params["audio_prompts"])

    try:
        while not stop_event.is_set():

            # -- Detect prompt changes --------------------------------
            try:
                prompts_changed = current_params["prompts"] != last_prompts
            except ValueError:
                prompts_changed = True

            try:
                audio_changed = any(
                    (a1[1] != a2[1]) or
                    (a1[0] is None) != (a2[0] is None) or
                    (a1[0] is not None and a2[0] is not None and not np.array_equal(a1[0], a2[0]))
                    for a1, a2 in zip(current_params["audio_prompts"], last_audio)
                )
            except (ValueError, TypeError):
                audio_changed = True

            # If something changed, launch compute_style() in the background.
            # generation continues with the previous style.
            if prompts_changed or audio_changed:
                last_prompts = list(current_params["prompts"])
                last_audio   = list(current_params["audio_prompts"])
                # Only submit a new Future if the previous one already finished.
                if _style_future is None or _style_future.done():
                    _style_future = _style_executor.submit(compute_style)

            # Update the style only if the Future is ready -- never blocking.
            if _style_future is not None and _style_future.done():
                _current_style = _style_future.result()
                _style_future  = None

            # -- Generate the chunk ------------------------------------
            # _current_style never changes mid-generation: it's only
            # updated between one chunk and the next.
            chunk, state = MRT.generate_chunk(
                state=state,
                style=_current_style,
                temperature=float(current_params["temperature"]),
                topk=int(current_params["topk"]),
                guidance_weight=float(current_params["guidance"]),
            )

            samples = chunk.samples.astype(np.float32)
            peak = np.abs(samples).max()
            if peak > 1.0:
                samples = samples / peak

            # Blocking put(): stalls once the queue is full (maxsize=8),
            # which rate-limits generation to playback speed.
            _audio_q.put(samples.tobytes())

    except Exception as e:
        print(f"[GENERATION ERROR] {e}")
        import traceback
        traceback.print_exc()

In [15]:
def start_generation():
    """Starts generation_loop on a new background thread, clearing any stale audio first."""
    global gen_thread
    stop_event.clear()
    # Empty the queue before starting, to avoid playing stale audio.
    while not _audio_q.empty():
        try: _audio_q.get_nowait()
        except: break
    gen_thread = threading.Thread(target=generation_loop, daemon=True)
    gen_thread.start()

def reset_state():
    """Stops generation_loop and resets all generation state (model state, pending style Future)."""
    global state, _style_future
    stop_event.set()
    # Empty the queue to unblock put() in the generation thread.
    while not _audio_q.empty():
        try: _audio_q.get_nowait()
        except: break
    # Cancel the pending Future, if any.
    if _style_future is not None and not _style_future.done():
        _style_future.cancel()
    _style_future = None
    state = None

### 4. JavaScript (AudioWorklet + ring buffer)

In [16]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Section 4 -- JavaScript (AudioWorklet + ring buffer)
#
# An AudioWorklet runs at 48kHz and reads from an internal ring buffer.
# The WebSocket fills the buffer as chunks arrive.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CLOUDSPACE_HOST = os.environ.get("LIGHTNING_CLOUDSPACE_HOST", "localhost")
AUDIO_WS_URL = f"wss://{AUDIO_WS_PORT}-{CLOUDSPACE_HOST}"

js_start = f"""
async () => {{
    console.log('▶ _startAudio called');

    // Close and recreate the AudioContext on every start (ensures a fresh worklet)
    if (window._workletNode) {{
        window._workletNode.disconnect();
        window._workletNode = null;
    }}
    if (window._aws) {{
        window._aws.close();
        window._aws = null;
    }}
    if (window._actx) {{
        try {{ await window._actx.close(); }} catch(e) {{}}
        window._actx = null;
    }}
    window._actx = new AudioContext({{sampleRate: 48000}});
    await window._actx.resume();

    // -- Load the AudioWorklet processor --
    if (!window._actx._workletLoaded) {{
        const code = `
class RingBufferProcessor extends AudioWorkletProcessor {{
    constructor() {{
        super();
        // Stereo interleaved float32 ring buffer -- ~5.4s at 48kHz
        this._buf       = new Float32Array(524288);
        this._writePos  = 0;
        this._readPos   = 0;
        this._available = 0;
        this._pendingRequest = false;

        this.port.onmessage = (e) => {{
            // Reset signal: clear the ring buffer
            if (e.data === 'flush') {{
                this._writePos  = 0;
                this._readPos   = 0;
                this._available = 0;
                this._pendingRequest = false;
                return;
            }}
            // Audio data: write it into the ring buffer
            const data = new Float32Array(e.data);
            if (this._available + data.length > this._buf.length) {{
                console.warn('[RingBuffer] overflow, dropping chunk');
                return;
            }}
            const space = this._buf.length - this._writePos;
            if (data.length <= space) {{
                this._buf.set(data, this._writePos);
                this._writePos += data.length;
            }} else {{
                this._buf.set(data.subarray(0, space), this._writePos);
                this._buf.set(data.subarray(space), 0);
                this._writePos = data.length - space;
            }}
            this._available += data.length;
            this._pendingRequest = false;
        }};
    }}

    process(inputs, outputs) {{
        const out = outputs[0];
        const left      = out[0];
        const right     = out[1];
        if (!left) return true; // safety check, output should always be stereo
        const blockSize = left.length; // 128 samples per call (~2.7ms)
        const needed    = blockSize * 2;

        if (this._available < needed) {{
            // Buffer empty: stay silent until new data arrives
            return true;
        }}
        for (let i = 0; i < blockSize; i++) {{
            left[i]  = this._buf[this._readPos];
            this._readPos = (this._readPos + 1) % this._buf.length;
            right[i] = this._buf[this._readPos];
            this._readPos = (this._readPos + 1) % this._buf.length;
        }}
        this._available -= needed;
        // Request the next chunk once less than 2.7s of audio remain
        if (this._available < 262144 && !this._pendingRequest) {{
            this._pendingRequest = true;
            this.port.postMessage({{type: 'need_chunk'}});
        }}
        return true;
    }}
}}
registerProcessor('ring-buffer-processor', RingBufferProcessor);
`;
        const blob = new Blob([code], {{type: 'application/javascript'}});
        const url  = URL.createObjectURL(blob);
        await window._actx.audioWorklet.addModule(url);
        URL.revokeObjectURL(url);
        window._actx._workletLoaded = true;
        console.log('✅ AudioWorklet loaded');
    }}

    // Create the worklet node and connect it to the audio output
    if (window._workletNode) {{
        window._workletNode.disconnect();
    }}
    window._workletNode = new AudioWorkletNode(window._actx, 'ring-buffer-processor', {{numberOfInputs: 0, numberOfOutputs: 1, outputChannelCount: [2]}});
    window._workletNode.connect(window._actx.destination);

    // Relay the 'need_chunk' signal to the WebSocket as 'ready'
    window._workletNode.port.onmessage = (e) => {{
        if (e.data && e.data.type === 'need_chunk') {{
            if (window._aws && window._aws.readyState === WebSocket.OPEN) {{
                window._aws.send('ready');
            }}
        }}
    }};

    // -- WebSocket ------------------------------------------------
    window._aws = new WebSocket('{AUDIO_WS_URL}');
    window._aws.binaryType = 'arraybuffer';

    window._aws.onopen = () => {{
        console.log('✅ Audio WS connected');
        const el = document.getElementById('audio-status');
        if (el) el.textContent = '🔊 Stream active';
    }};

    window._aws.onmessage = (e) => {{
        if (!window._workletNode) return; // worklet not ready yet

        // FLUSH signal: clear the ring buffer
        if (e.data instanceof ArrayBuffer && e.data.byteLength === 5) {{
            const txt = new TextDecoder().decode(e.data);
            if (txt === 'FLUSH') {{
                window._workletNode.port.postMessage('flush');
                console.log('Audio Flush');
                return;
            }}
        }}
        // Transfer the ArrayBuffer to the worklet (zero-copy, no memory copy)
        window._workletNode.port.postMessage(e.data, [e.data]);
    }};

    window._aws.onerror = (e) => {{
        console.error('❌ Audio WS error', e);
        const el = document.getElementById('audio-status');
        if (el) el.textContent = '❌ WebSocket error';
    }};

    window._aws.onclose = (e) => {{
        console.log('Audio WS closed, code:', e.code);
        const el = document.getElementById('audio-status');
        if (el) el.textContent = '⏹ Stream stopped';
    }};
}}
"""

In [17]:
js_stop = """
() => {
    console.log('⏹ _stopAudio called');
    if (window._aws) window._aws.close();
    if (window._workletNode) {
        window._workletNode.port.postMessage('flush');
        window._workletNode.disconnect();
        window._workletNode = null;
    }
    const el = document.getElementById('audio-status');
    if (el) el.textContent = '⏹ Stream stopped';
}
"""

### 5. Gradio UI

In [22]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Section 5 -- Gradio UI
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

with gr.Blocks() as demo:
    gr.Markdown("## Magenta RT")

    with gr.Accordion("Sampling options", open=True):
        with gr.Row():
            temperature = gr.Slider(0.0, 4.0,  value=current_params["temperature"], step=0.01, label="Temperature",
                info="Creativity/chaos. Low values (0.9) = stable; high values (1.5) = experimental.")
            topk        = gr.Slider(1, 1024,   value=current_params["topk"],        step=1,    label="Top-k",
                info="Size of the sampling vocabulary. Low = coherent; high = varied but noisier.")
            guidance    = gr.Slider(0.0, 10.0, value=current_params["guidance"],    step=0.01, label="Guidance",
                info="Adherence to the prompts. High = faithful to the style; low = more freedom.")

    with gr.Accordion("Text prompts", open=True):
        gr.Markdown("Combine multiple styles with weights. Try *synthwave* + *flamenco guitar*!")
        text_boxes   = []
        text_weights = []
        for i in range(NUM_TEXT_PROMPTS):
            with gr.Row():
                t = gr.Textbox(value="synthwave" if i == 0 else "",
                               placeholder="e.g. jazz piano, lo-fi, orchestral...",
                               label=f"Prompt {i+1}", scale=3)
                w = gr.Slider(0.0, 1.0, value=1.0 if i == 0 else 0.0,
                              step=0.01, label="Weight", scale=1)
                text_boxes.append(t)
                text_weights.append(w)

    with gr.Accordion("Audio prompts", open=False):
        gr.Markdown("Upload an audio file as a style reference (max 10s, formats: .wav .mp3 .ogg).")
        audio_uploads = []
        audio_weights = []
        for i in range(NUM_AUDIO_PROMPTS):
            with gr.Row():
                initial_value = None
                if initial_audio_prompts[i][0] is not None:
                    preview_data = librosa.resample(initial_audio_prompts[i][0], orig_sr=16000, target_sr=44100)
                    initial_value = (44100, preview_data)

                a = gr.Audio(value=initial_value, type="numpy", label=f"Audio prompt {i+1}", scale=3)
                w = gr.Slider(0.0, 1.0, value=0.0, step=0.01, label="Weight", scale=1)
                audio_uploads.append(a)
                audio_weights.append(w)

    with gr.Row():
        btn   = gr.Button("▶ Generate", variant="primary")
        reset = gr.Button("↺ Reset")

    gr.HTML('<div id="audio-status" style="padding:8px; color:#888;">▷ Press Generate to start</div>')

    temperature.change(update_temp,    inputs=temperature)
    topk.change(update_topk,           inputs=topk)
    guidance.change(update_guidance,   inputs=guidance)

    for i in range(NUM_TEXT_PROMPTS):
        text_boxes[i].change(make_text_updater(i),          inputs=text_boxes[i])
        text_weights[i].change(make_text_weight_updater(i), inputs=text_weights[i])

    for i in range(NUM_AUDIO_PROMPTS):
        audio_uploads[i].change(make_audio_updater(i),        inputs=audio_uploads[i])
        audio_weights[i].change(make_audio_weight_updater(i), inputs=audio_weights[i])

    btn.click(fn=start_generation).then(fn=None, js=js_start)
    reset.click(fn=reset_state).then(fn=None, js=js_stop)

    def sync_ui_from_params():
        """Gradio Timer callback (every 0.5s): pushes current_params back into
        the UI components, so changes coming from the prompt websocket
        (e.g. emotion-driven updates) are reflected in the interface."""
        prompts       = current_params["prompts"]
        audio_prompts = current_params["audio_prompts"]
        result = []
        # Update the sampling sliders.
        result.append(gr.update(value=current_params["temperature"]))
        result.append(gr.update(value=current_params["topk"]))
        result.append(gr.update(value=current_params["guidance"]))
        # Update the text-prompt strings.
        for i in range(NUM_TEXT_PROMPTS):
            text, _ = prompts[i] if i < len(prompts) else ("", 0.0)
            result.append(gr.update(value=text))
        # Update the text-prompt weights.
        for i in range(NUM_TEXT_PROMPTS):
            _, weight = prompts[i] if i < len(prompts) else ("", 0.0)
            result.append(gr.update(value=weight))
        # FIX: don't update the audio data -- only the weight.
        # Updating the data caused a float32->int16->float32 round-trip on
        # every tick, making audio_changed constantly True.
        for i in range(NUM_AUDIO_PROMPTS):
            result.append(gr.update())
        # Update the audio-prompt weights.
        for i in range(NUM_AUDIO_PROMPTS):
            _, weight = audio_prompts[i] if i < len(audio_prompts) else (None, 0.0)
            result.append(gr.update(value=weight))
        return result

    timer = gr.Timer(0.5)
    timer.tick(
        sync_ui_from_params,
        outputs=[temperature, topk, guidance]
                + text_boxes + text_weights
                + audio_uploads + audio_weights
    )

gr.close_all()
subprocess.run('lsof -ti:7860 | xargs kill -9', shell=True, capture_output=True)
time.sleep(1)
demo.launch(server_port=7860, show_error=True)

/teamspace/studios/this_studio/magenta_venv/lib/python3.12/site-packages/gradio/processing_utils.py:698: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 4813)
[Audio WS] Client connected: ('130.211.236.75', 41735)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 44136)
[WS] FLUSH, flushed 8 chunks
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 5842)
[WS] FLUSH, flushed 8 chunks
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 7908)
[Audio WS] Client disconnected
[WS] FLUSH, flushed 8 chunks
[WS] Client disconnected
[Audio WS] Client connected: ('130.211.236.75', 10514)
[WS] Client connected: ('130.211.236.75', 10617)
[WS] FLUSH, flushed 0 chunks
[WS] Client disconnected
[WS] Client connected: ('34.71.39.87', 62503)
[WS] Client disconnected
[WS] Client connected: ('34.71.39.87', 62149)
[WS] Client disconnected
[WS] Client connected: ('34.71.39.87', 63431)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 16128)
[WS] FLUSH, flushed 8 chunks
[WS] Client disconnected
[WS] Client connected: ('130.211.23

# Prompt Websocket
The communication channel for receiving user text from the frontend to the Python/Jupyter backend. It carries the raw prompt and triggers music generation.

In [19]:
WS_PORT = 9002

# Global references to the prompt-websocket server's event loop and thread.
_ws_loop = None
_ws_thread = None

In [20]:
def stop_ws_server():
    """Stops the currently running prompt-websocket server, if any."""
    global _ws_loop, _ws_thread
    if _ws_loop is not None and _ws_loop.is_running():
        _ws_loop.call_soon_threadsafe(_ws_loop.stop)
    if _ws_thread is not None and _ws_thread.is_alive():
        _ws_thread.join(timeout=3)
    _ws_loop = None
    _ws_thread = None


async def handle_connection(websocket):
    """Handles one prompt-websocket connection.

    Each incoming message is a JSON object mapping a prompt text (or an
    audio-prompt key like "_anger_audio_") to a weight in [0.0, 1.0].
    A "__flush__" key instead triggers an immediate audio-queue flush.
    """
    print(f"[WS] Client connected: {websocket.remote_address}")
    try:
        async for message in websocket:
                try:
                    updates = json.loads(message)
                    # If the client sent "__flush__", empty the audio queue
                    # so playback jumps ahead immediately.
                    if "__flush__" in updates:
                        flushed = 0
                        try:
                            while not _audio_q.empty():
                                try:
                                    _audio_q.get_nowait()
                                    flushed +=1
                                except:
                                    break
                            _flush_flag.set()
                            print(f"[WS] FLUSH, flushed {flushed} chunks")
                        except:
                            pass
                        continue

                    for prompt_text, weight in updates.items():
                        if prompt_text in audio_key_to_index:
                            idx = audio_key_to_index[prompt_text]

                            raw_weight = float(weight)
                            # Weight is mapped from [0.0, 1.0] to [0.0, 0.5]
                            actual_weight = raw_weight * 0.5

                            # Keep the current audio data and update only the weight
                            current_audio_data = current_params["audio_prompts"][idx][0]
                            current_params["audio_prompts"][idx] = (current_audio_data, round(actual_weight, 2))

                        else:
                            for i, (text, _) in enumerate(current_params["prompts"]):
                                if text == prompt_text:
                                    current_params["prompts"][i] = (prompt_text, float(weight))
                                    break

                except Exception as e:
                    print(f"[WS] Error: {e}")
    except websockets.exceptions.ConnectionClosedError:
        pass
    finally:
        print(f"[WS] Client disconnected")


async def ws_server():
    async with websockets.serve(handle_connection, "0.0.0.0", WS_PORT, reuse_address = True):
        print(f"[WS] Listening on :{WS_PORT}")
        await asyncio.Future()  # runs forever

def start_ws_server():
    global _ws_loop
    _ws_loop = asyncio.new_event_loop()
    asyncio.set_event_loop(_ws_loop)
    _ws_loop.run_until_complete(ws_server())

In [21]:
import subprocess, time
subprocess.run('lsof -ti:9002 | xargs kill -9', shell=True, capture_output=True)
time.sleep(3)

# Stop any previous server instance before starting a new one.
stop_ws_server()

_ws_thread = threading.Thread(target=start_ws_server, daemon=True)
_ws_thread.start()

[WS] Listening on :9002
[WS] Client connected: ('130.211.236.75', 4896)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 14623)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 13069)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 5387)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 6405)
[WS] Client disconnected
[WS] Client connected: ('130.211.236.75', 7419)
